# Data Cleaning & Preprocessing

**Purpose:** This notebook processes and analyzes food label data for allergen detection.


## Overview
Cleans and standardizes extracted ingredient data.

## Workflow
1. Remove duplicates and invalid entries
2. Normalize ingredient names
3. Handle missing values
4. Validate data quality


In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/interim/extracted_raw.csv")

In [2]:
clean_df = df.copy()
clean_df = clean_df.dropna(subset=["ingredients_text_en"])
clean_df["ingredients_text_en"] = clean_df["ingredients_text_en"].str.lower().str.strip()

def strip_html(text):
    return re.sub(r'<[^>]+>', '', text)

clean_df["ingredients_text_en"] = clean_df["ingredients_text_en"].apply(strip_html)

In [3]:
clean_df = clean_df[clean_df["ingredients_text_en"].str.len() > 15]
clean_df = clean_df[~clean_df["ingredients_text_en"].str.contains("http|www|barcode", na=False)]
print(f"After length/URL filter: {len(clean_df)} rows")

After length/URL filter: 1058 rows


In [4]:
def normalize_text(text):
    # Remove parentheses containing only digits and % (e.g., (30%))
    text = re.sub(r"\(\d+%\)", "", text)
    # Replace other parentheses with commas
    text = re.sub(r"[()]", ",", text)
    # Replace semicolons with commas
    text = text.replace(";", ",")
    # Remove standalone percentages (e.g., 30% but not "30% milk")
    text = re.sub(r"\b\d+%\b", "", text)
    # Keep letters (including accented), spaces, commas, hyphens, apostrophes, ampersands, slashes
    text = re.sub(r"[^a-zà-ÿ\s,\-'\&/]", "", text)
    # Collapse multiple commas and spaces
    text = re.sub(r",+", ",", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_df["ingredients_cleaned"] = clean_df["ingredients_text_en"].apply(normalize_text)

In [5]:
def tokenize(text):
    return [t.strip() for t in text.split(",") if t.strip()]

# Filter tokens that are likely from HTML or non-ingredients
def filter_tokens(tokens):
    stop_tokens = {"span", "class", "allergen", "div", "id", "style", "\\", ""}
    return [t for t in tokens if t not in stop_tokens and len(t) > 1]

clean_df["ingredient_tokens_raw"] = clean_df["ingredients_cleaned"].apply(tokenize)
clean_df["ingredient_tokens"] = clean_df["ingredient_tokens_raw"].apply(filter_tokens)

# Drop rows that become empty after filtering
clean_df = clean_df[clean_df["ingredient_tokens"].apply(len) > 0]
print(f"After token filtering: {len(clean_df)} rows")

After token filtering: 1057 rows


In [6]:
# Optional: filter by category keywords – but this may discard relevant products.
# We'll keep it commented out; you can uncomment and adjust if needed.
# keywords = ["snack", "cracker", "beverage", "biscuit", "wafer", "dairy", "milk",
#             "juice", "soda", "chips", "cookies", "candy", "chocolate", "nut"]
# clean_df = clean_df[clean_df["categories"].str.contains("|".join(keywords), case=False, na=False)]

In [7]:
clean_df = clean_df.drop_duplicates(subset=["code"])
print(f"Final cleaned rows: {len(clean_df)}")

Final cleaned rows: 1057


In [8]:
clean_df.to_csv("../data/processed/cleaned_dataset.csv", index=False)
print("Saved to ../data/processed/cleaned_dataset.csv")

Saved to ../data/processed/cleaned_dataset.csv
